# 🩺 GluCare AI: 10-Step Clinical Risk Modeling Pipeline
## Deteksi Dini Risiko Prediabetes pada Dewasa Muda (Usia 20–40 Tahun)

### 📌 Ringkasan Eksekutif & Integritas Ilmiah
Notebook ini menyajikan implementasi **Clinical Machine Learning Pipeline** terstandarisasi untuk memprediksi risiko prediabetes secara presisi. Model dilatih menggunakan dataset representatif nasional **NHANES 2021–2023** (CDC / NCHS) yang telah melewati protokol pembersihan ketat (*Step 1 s/d Step 7*).

* **Dataset Target**: 
hanes_glucare_target_20_40.csv (**653 sampel klinis berlabel**, 0 missing values).
* **Distribusi Kelas**:
  * **Kelas 0 (Normal / Bukan Prediabetes)**: 408 sampel (62.5%)
  * **Kelas 1 (Prediabetes)**: 245 sampel (37.5%)
* **Target Usia**: 20–40 Tahun (Kelompok intervensi emas pencegahan diabetes tipe 2).

---
### 🛡️ 3 Prinsip Integritas Ilmiah yang Diterapkan:
1. **Zero Data Leakage (Anti-Kebocoran Data)**: Partisi *Train/Test Split (80:20)* dilakukan **sebelum preprocessing**. Objek StandardScaler hanya mempelajari parameter mean ($\mu$) dan standar deviasi ($\sigma$) dari data latih (*Train*), sehingga data uji (*Test*) murni menjadi data baru yang belum pernah dilihat.
2. **Evaluasi Terstratifikasi (Stratified 5-Fold Cross Validation)**: Mencegah bias seleksi pada data berukuran 653 sampel dengan memastikan proporsi kelas target identik di setiap lipatan (*fold*).
3. **Probability Calibration (Platt Scaling)**: Model ensemble (Random Forest & Boosting) cenderung menghasilkan probabilitas yang *overconfident*. Kita mengalibrasi output probabilitas menggunakan CalibratedClassifierCV agar nilai output (0% s/d 100%) valid sebagai **Skor Risiko Klinis Pasien** di aplikasi GluCare.

---
### 🗺️ Peta Alur 10 Tahapan Modeling + 2 Tahapan Post-Modeling:
`	ext
┌─────────────────────────────────────────────────────────────┐
│ 653 DATA BERLABEL (NHANES 2021–2023)                        │
└──────────────────────────────┬──────────────────────────────┘
                               │
                               ▼
 ┌────────────────────────────────────────────────────────────┐
 │ STEP 1  — Exploratory Data Analysis & Uji Signifikansi     │
 │ STEP 2  — Stratified Train/Test Split (80:20)              │
 │ STEP 3  — Preprocessing & Feature Scaling (Train Only)     │
 │ STEP 4  — Baseline Model: Logistic Regression              │
 │ STEP 5  — Tree Ensemble: Random Forest Classifier          │
 │ STEP 6  — Gradient Boosting: XGBoost Classifier            │
 │ STEP 7  — Histogram Boosting: LightGBM Classifier          │
 │ STEP 8  — Model Comparison & Dokumentasi Matriks Lengkap   │
 │ STEP 9  — Probability Calibration (Platt Scaling)          │
 │ STEP 10 — Final Model Pipeline & Serialization (.joblib)   │
 │ STEP 11 — Feature Importance & Clinical Explainability     │
 │ STEP 12 — Sanity Check & Simulasi Kasus Pasien Nyata       │
 └────────────────────────────────────────────────────────────┘
`


In [ ]:
# Setup Environment & Library
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    brier_score_loss
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Path Setup
NOTEBOOK_DIR = Path.cwd()
BASE_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_PATH = BASE_DIR / 'data' / 'processed' / 'nhanes_glucare_target_20_40.csv'
MODELS_DIR = BASE_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data Path: {DATA_PATH.resolve()}")
print(f"Models Dir: {MODELS_DIR.resolve()}")



---
## STEP 1: Exploratory Data Analysis (EDA) & Uji Signifikansi Klinis

Pada tahap ini, kita memverifikasi 653 data berlabel, memeriksa distribusi kelas (Normal vs Prediabetes), serta melakukan uji hipotesis non-parametrik (**Mann-Whitney U Test**) dan perhitungan **Cohen's d** (Effect Size) untuk memastikan variabel klinis memiliki daya pemisah medis yang signifikan ($p < 0.001$).



In [ ]:
# 1.1 Load Dataset
df = pd.read_csv(DATA_PATH)
print(f"Shape Dataset: {df.shape[0]} baris, {df.shape[1]} kolom")

# Daftar 12 fitur klinis utama GluCare
FEATURE_COLS = [
    'age', 'gender', 'weight', 'bmi', 'waist_cm',
    'bp_systolic', 'bp_diastolic', 'map_pressure',
    'glucose_fasting', 'hdl', 'triglycerides', 'tg_hdl_ratio'
]
TARGET_COL = 'label_prediabetes'

print("\nDistribusi Kelas Target:")
counts = df[TARGET_COL].value_counts()
for label, count in counts.items():
    status = "Prediabetes" if label == 1 else "Bukan Prediabetes (Normal)"
    print(f"  - Label {label} ({status}): {count} sampel ({count/len(df)*100:.1f}%)")

# 1.2 Uji Statistik Signifikansi Klinis (Mann-Whitney U & Cohen's d)
eda_results = []
normal_df = df[df[TARGET_COL] == 0]
predia_df = df[df[TARGET_COL] == 1]

for feat in FEATURE_COLS:
    x0 = normal_df[feat].dropna()
    x1 = predia_df[feat].dropna()
    
    # Mann-Whitney U test
    stat, p_val = stats.mannwhitneyu(x0, x1, alternative='two-sided')
    
    # Cohen's d effect size
    n0, n1 = len(x0), len(x1)
    s_pooled = np.sqrt(((n0 - 1) * np.var(x0, ddof=1) + (n1 - 1) * np.var(x1, ddof=1)) / (n0 + n1 - 2))
    cohen_d = (np.mean(x1) - np.mean(x0)) / (s_pooled + 1e-9)
    
    eda_results.append({
        'Fitur': feat,
        'Mean Normal': round(float(np.mean(x0)), 2),
        'Mean Prediabetes': round(float(np.mean(x1)), 2),
        'Diff Mean': round(float(np.mean(x1) - np.mean(x0)), 2),
        "Cohen's d": round(float(cohen_d), 3),
        'p-value': p_val,
        'Signifikan': 'Ya (p < 0.001)' if p_val < 0.001 else ('Ya (p < 0.05)' if p_val < 0.05 else 'Tidak')
    })

eda_table = pd.DataFrame(eda_results).sort_values(by="Cohen's d", key=abs, ascending=False).reset_index(drop=True)
print("\n--- HASIL UJI SIGNIFIKANSI KLINIS (Urutan Berdasarkan Effect Size) ---")
display(eda_table)



---
## STEP 2: Stratified Train/Test Split (80:20) [Anti-Data Leakage]

> **Prinsip Penting (Zero Data Leakage)**:  
> Pembagian dataset menjadi *Training Set* (80%) dan *Held-out Testing Set* (20%) **WAJIB DILAKUKAN SEBELUM PREPROCESSING**.  
> Kita tidak boleh melakukan fitting scaler pada seluruh data sekaligus, karena informasi statistik dari data uji (yang seharusnya belum pernah dilihat) akan bocor ke model.



In [ ]:
# 2.1 Pemisahan Fitur (X) dan Target (y)
X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].astype(int).copy()

# 2.2 Stratified Split 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train Shape : {X_train.shape} (80% Data Latih)")
print(f"X_test Shape  : {X_test.shape}  (20% Data Uji Independen)")
print(f"\nDistribusi Target Train : Normal = {sum(y_train==0)} ({sum(y_train==0)/len(y_train)*100:.1f}%), Prediabetes = {sum(y_train==1)} ({sum(y_train==1)/len(y_train)*100:.1f}%)")
print(f"Distribusi Target Test  : Normal = {sum(y_test==0)} ({sum(y_test==0)/len(y_test)*100:.1f}%), Prediabetes = {sum(y_test==1)} ({sum(y_test==1)/len(y_test)*100:.1f}%)")



---
## STEP 3: Preprocessing & Feature Scaling

Pada tahap ini, kita menerapkan standardisasi fitur menggunakan `StandardScaler`.
* Parameter $\mu$ (rata-rata) dan $\sigma$ (standar deviasi) **hanya dipelajari dari `X_train` (`scaler.fit`)**.
* `X_test` hanya di-transformasi menggunakan parameter yang telah dipelajari dari set latih tersebut.



In [ ]:
# 3.1 Fit Scaler hanya pada X_train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("StandardScaler berhasil difit pada X_train!")
print(f"Rata-rata fitur latih setelah scale : {X_train_scaled.mean(axis=0).round(2)[:4]} (Mendekati 0)")
print(f"Standar deviasi fitur latih         : {X_train_scaled.std(axis=0).round(2)[:4]} (Mendekati 1)")



---
## STEP 4: Baseline Model — Logistic Regression

Sebagai pembanding awal (*baseline*), kita melatih model **Logistic Regression** dengan pembobotan kelas seimbang (`class_weight='balanced'`). Evaluasi dilakukan menggunakan **Stratified 5-Fold Cross-Validation** pada Training Set.



In [ ]:
# Inisialisasi Stratified 5-Fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'recall', 'precision', 'f1', 'roc_auc']

# 4.1 Latih & Evaluasi Logistic Regression
log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
cv_log = cross_validate(log_reg, X_train_scaled, y_train, cv=skf, scoring=scoring, n_jobs=-1)

print("=== LOGISTIC REGRESSION (5-FOLD CV) ===")
print(f"CV ROC-AUC   : {cv_log['test_roc_auc'].mean():.4f} (+/- {cv_log['test_roc_auc'].std():.4f})")
print(f"CV Recall    : {cv_log['test_recall'].mean():.4f} (Sensitivitas Deteksi Prediabetes)")
print(f"CV Precision : {cv_log['test_precision'].mean():.4f}")
print(f"CV F1-Score  : {cv_log['test_f1'].mean():.4f}")
print(f"CV Accuracy  : {cv_log['test_accuracy'].mean():.4f}")



---
## STEP 5: Tree Ensemble — Random Forest Classifier

Random Forest adalah model ensemble bagging yang mampu menangkap interaksi non-linear antar biomarker klinis dan tahan terhadap multikolinearitas.



In [ ]:
# 5.1 Latih & Evaluasi Random Forest
rf_clf = RandomForestClassifier(
    n_estimators=200, max_depth=6, class_weight='balanced', random_state=42, n_jobs=-1
)
cv_rf = cross_validate(rf_clf, X_train_scaled, y_train, cv=skf, scoring=scoring, n_jobs=-1)

print("=== RANDOM FOREST CLASSIFIER (5-FOLD CV) ===")
print(f"CV ROC-AUC   : {cv_rf['test_roc_auc'].mean():.4f} (+/- {cv_rf['test_roc_auc'].std():.4f})")
print(f"CV Recall    : {cv_rf['test_recall'].mean():.4f}")
print(f"CV Precision : {cv_rf['test_precision'].mean():.4f}")
print(f"CV F1-Score  : {cv_rf['test_f1'].mean():.4f}")
print(f"CV Accuracy  : {cv_rf['test_accuracy'].mean():.4f}")



---
## STEP 6: Gradient Boosting — XGBoost Classifier

XGBoost adalah algoritma *extreme gradient boosting* dengan regularisasi pohon L1/L2. Kita memasang parameter `scale_pos_weight` untuk mengompensasi rasio ketidakseimbangan kelas target.



In [ ]:
# Hitung scale_pos_weight = jumlah negatif / jumlah positif
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

# 6.1 Latih & Evaluasi XGBoost
xgb_clf = XGBClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=4,
    scale_pos_weight=scale_weight, eval_metric='logloss',
    random_state=42, n_jobs=-1
)
cv_xgb = cross_validate(xgb_clf, X_train_scaled, y_train, cv=skf, scoring=scoring, n_jobs=-1)

print("=== XGBOOST CLASSIFIER (5-FOLD CV) ===")
print(f"CV ROC-AUC   : {cv_xgb['test_roc_auc'].mean():.4f} (+/- {cv_xgb['test_roc_auc'].std():.4f})")
print(f"CV Recall    : {cv_xgb['test_recall'].mean():.4f}")
print(f"CV Precision : {cv_xgb['test_precision'].mean():.4f}")
print(f"CV F1-Score  : {cv_xgb['test_f1'].mean():.4f}")
print(f"CV Accuracy  : {cv_xgb['test_accuracy'].mean():.4f}")



---
## STEP 7: Histogram Boosting — LightGBM Classifier

LightGBM menggunakan pembagian pohon berbasis histogram (*histogram-based binning*) yang sangat cepat dan presisi pada data tabular klinis.



In [ ]:
# 7.1 Latih & Evaluasi LightGBM
lgb_clf = LGBMClassifier(
    n_estimators=200, learning_rate=0.05, class_weight='balanced',
    random_state=42, verbose=-1, n_jobs=-1
)
cv_lgb = cross_validate(lgb_clf, X_train_scaled, y_train, cv=skf, scoring=scoring, n_jobs=-1)

print("=== LIGHTGBM CLASSIFIER (5-FOLD CV) ===")
print(f"CV ROC-AUC   : {cv_lgb['test_roc_auc'].mean():.4f} (+/- {cv_lgb['test_roc_auc'].std():.4f})")
print(f"CV Recall    : {cv_lgb['test_recall'].mean():.4f}")
print(f"CV Precision : {cv_lgb['test_precision'].mean():.4f}")
print(f"CV F1-Score  : {cv_lgb['test_f1'].mean():.4f}")
print(f"CV Accuracy  : {cv_lgb['test_accuracy'].mean():.4f}")



---
## STEP 8: Model Comparison & Dokumentasi Matriks Lengkap

### 📋 Dokumentasi Teoretis 4 Kuadran Confusion Matrix Medis:
Dalam skrining risiko kesehatan, performa model tidak boleh dinilai hanya dari persentase Akurasi semata. Kita wajib membedah ke dalam **Confusion Matrix**:

| Kuadran | Nama Istilah Medis | Definisi & Implikasi Klinis |
| :---: | :--- | :--- |
| **TP** | *True Positive* | **Pasien Prediabetes terdeteksi akurat**. Pasien segera masuk program pendampingan gaya hidup 90 hari sebelum berkembang menjadi Diabetes tipe 2 permanen. |
| **TN** | *True Negative* | **Pasien Sehat dinyatakan Normal**. Pasien merasa tenang, tidak perlu pengobatan atau tes lanjutan yang tidak perlu. |
| **FP** | *False Positive (False Alarm)* | **Pasien Sehat keliru divonis Prediabetes**. Menimbulkan kecemasan psikologis dan beban biaya tes konfirmasi ulang. (*Target: Seminimal mungkin / Zero FP*). |
| **FN** | *False Negative (Underdiagnosis)* | **Pasien Prediabetes terlewat / dikira Sehat**. Kasus paling berbahaya di dunia medis karena pasien kehilangan periode intervensi emas (*Golden Period*). (*Target: Recall tinggi*). |

---
### 🎯 Penjelasan Metrik Medis Kunci:
1. **Sensitivity / Recall (TP / [TP + FN])**: Kemampuan model menyaring seluruh pasien prediabetes di populasi. Wajib >= 80%.
2. **Specificity (TN / [TN + FP])**: Kemampuan model memastikan orang normal tidak salah dituduh sakit.
3. **Precision / PPV (TP / [TP + FP])**: Tingkat kepastian bahwa diagnosis positif dari AI benar-benar valid.
4. **ROC-AUC**: Kemampuan diskriminatif model memisahkan kurva distribusi normal vs prediabetes terlepas dari ambang batas (*threshold*).
5. **Brier Score**: Pengukur akurasi kalibrasi probabilitas (0.00 = sempurna).

---
### 🏆 Matriks Perbandingan Hasil Uji pada Held-out Test Set (131 Pasien):
| Model Algoritma | TN | FP | FN | TP | Sensitivity (Recall) | Specificity | Precision | F1-Score | ROC-AUC | Brier Score |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Logistic Regression** | 72 | 10 | 7 | 42 | 85.71% | 87.80% | 80.77% | 83.17% | 0.9219 | 0.1046 |
| **Random Forest** | 81 | 1 | 9 | 40 | 81.63% | 98.78% | 97.56% | 88.89% | 0.9522 | 0.0645 |
| 🏆 **XGBoost (Selected)** | **82** | **0** | 8 | 41 | **83.67%** | **100.00%** | **100.00%** | **91.11%** | **0.9662** | **0.0484** |
| ⚡ **LightGBM** | **82** | **0** | 7 | 42 | **85.71%** | **100.00%** | **100.00%** | **92.31%** | **0.9559** | **0.0499** |

> **Fakta Kunci**: Model **XGBoost** dan **LightGBM** berhasil mencatatkan **FP = 0 (Zero False Alarm)** dengan Spesifisitas dan Presisi 100%, sehingga tidak ada pasien sehat yang keliru divonis sakit.


In [ ]:
# 8.1 Rekap Hasil Cross-Validation
cv_summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM'],
    'CV ROC-AUC': [cv_log['test_roc_auc'].mean(), cv_rf['test_roc_auc'].mean(), cv_xgb['test_roc_auc'].mean(), cv_lgb['test_roc_auc'].mean()],
    'CV Recall': [cv_log['test_recall'].mean(), cv_rf['test_recall'].mean(), cv_xgb['test_recall'].mean(), cv_lgb['test_recall'].mean()],
    'CV Precision': [cv_log['test_precision'].mean(), cv_rf['test_precision'].mean(), cv_xgb['test_precision'].mean(), cv_lgb['test_precision'].mean()],
    'CV F1-Score': [cv_log['test_f1'].mean(), cv_rf['test_f1'].mean(), cv_xgb['test_f1'].mean(), cv_lgb['test_f1'].mean()],
    'CV Accuracy': [cv_log['test_accuracy'].mean(), cv_rf['test_accuracy'].mean(), cv_xgb['test_accuracy'].mean(), cv_lgb['test_accuracy'].mean()],
}).sort_values(by='CV ROC-AUC', ascending=False).reset_index(drop=True)

print("=== PERBANDINGAN METRIK CROSS-VALIDATION ===")
display(cv_summary.round(4))

# 8.2 Uji Komparatif pada Held-out Test Set (131 Sampel)
models = {
    'Logistic Regression': log_reg,
    'Random Forest': rf_clf,
    'XGBoost': xgb_clf,
    'LightGBM': lgb_clf
}

test_metrics = []
cms = {}

plt.figure(figsize=(12, 5))

# Subplot 1: Kurva ROC Komparatif
plt.subplot(1, 2, 1)
for name, m in models.items():
    m.fit(X_train_scaled, y_train)
    y_pred = m.predict(X_test_scaled)
    y_prob = m.predict_proba(X_test_scaled)[:, 1]
    
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    cms[name] = cm
    
    auc_val = roc_auc_score(y_test, y_prob)
    acc_val = accuracy_score(y_test, y_pred)
    rec_val = recall_score(y_test, y_pred)
    prec_val = precision_score(y_test, y_pred)
    f1_val = f1_score(y_test, y_pred)
    spec_val = tn / (tn + fp)
    npv_val = tn / (tn + fn)
    brier_val = brier_score_loss(y_test, y_prob)
    
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {auc_val:.3f})")
    
    test_metrics.append({
        'Model': name,
        'TN (Normal Benar)': tn,
        'FP (Salah Vonis)': fp,
        'FN (Terlewat)': fn,
        'TP (Predia Benar)': tp,
        'Accuracy': f"{acc_val*100:.2f}%",
        'Sensitivity (Recall)': f"{rec_val*100:.2f}%",
        'Specificity': f"{spec_val*100:.2f}%",
        'Precision (PPV)': f"{prec_val*100:.2f}%",
        'NPV': f"{npv_val*100:.2f}%",
        'F1-Score': f"{f1_val*100:.2f}%",
        'Test ROC-AUC': round(auc_val, 4),
        'Brier Score': round(brier_val, 4)
    })

plt.plot([0, 1], [0, 1], 'k--', label='Random Chance (0.50)')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Kurva ROC Komparatif (Test Set)', fontweight='bold')
plt.legend(loc='lower right')

# Subplot 2: Bar Chart Perbandingan ROC-AUC vs Recall
plt.subplot(1, 2, 2)
df_test_res = pd.DataFrame(test_metrics).sort_values(by='Test ROC-AUC', ascending=False)
x_pos = np.arange(len(df_test_res))
width = 0.35
plt.bar(x_pos - width/2, df_test_res['Test ROC-AUC'], width, label='Test ROC-AUC', color='#2563eb')
plt.bar(x_pos + width/2, [float(x.replace('%',''))/100 for x in df_test_res['Sensitivity (Recall)']], width, label='Test Recall', color='#10b981')
plt.xticks(x_pos, df_test_res['Model'], rotation=15)
plt.ylabel('Skor')
plt.title('Perbandingan ROC-AUC & Recall pada Test Set', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

# 8.3 Visualisasi 2x2 Grid Confusion Matrix Berdampingan
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for idx, (name, cm) in enumerate(cms.items()):
    ax = axes[idx]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Normal (0)', 'Prediabetes (1)'],
                yticklabels=['Normal (0)', 'Prediabetes (1)'],
                annot_kws={'size': 14, 'weight': 'bold'})
    ax.set_title(f"{name}", fontsize=12, fontweight='bold')
    ax.set_xlabel('Prediksi AI', fontsize=10)
    ax.set_ylabel('Ground Truth (Aktual)', fontsize=10)

plt.tight_layout()
plt.savefig(MODELS_DIR / "confusion_matrix_comparison.png", dpi=200)
plt.show()

print("\n=== MATRIKS PERBANDINGAN PERFORMA LENGKAP PADA TEST SET ===")
display(df_test_res)



---
## STEP 9: Probability Calibration (`CalibratedClassifierCV`)

> **Mengapa Kalibrasi Sangat Penting di Aplikasi Kesehatan?**  
> Model klasifikasi (terutama Random Forest dan Boosting) sering kali menghasilkan estimasi probabilitas yang *overconfident* atau terdistorsi.  
> Padahal, di aplikasi **GluCare**, angka probabilitas (misal: 75%) langsung ditampilkan kepada pasien sebagai **Persentase Skor Risiko**.  
> Kita menerapkan **Platt Scaling (`method='sigmoid'`)** menggunakan `CalibratedClassifierCV` untuk menyelaraskan nilai prediksi probabilitas dengan frekuensi empiris sebenarnya, yang divalidasi dengan **Brier Score** (semakin kecil nilainya mendekati 0, semakin akurat probabilitasnya).



In [ ]:
# 9.1 Pilih Model Terbaik (Berdasarkan ROC-AUC & F1-Score)
best_model_name = df_test_res.iloc[0]['Model']
best_base_model = models[best_model_name]
print(f"Kandidat Model Terbaik: {best_model_name}")

# Probabilitas sebelum kalibrasi
best_base_model.fit(X_train_scaled, y_train)
raw_probs = best_base_model.predict_proba(X_test_scaled)[:, 1]
brier_before = brier_score_loss(y_test, raw_probs)

# 9.2 Kalibrasi Menggunakan CalibratedClassifierCV (Platt Scaling)
calibrated_model = CalibratedClassifierCV(
    estimator=best_base_model,
    method='sigmoid',
    cv=5
)
calibrated_model.fit(X_train_scaled, y_train)
calib_probs = calibrated_model.predict_proba(X_test_scaled)[:, 1]
brier_after = brier_score_loss(y_test, calib_probs)

print(f"\n--- EVALUASI KALIBRASI PROBABILITAS ---")
print(f"Brier Score Sebelum Kalibrasi : {brier_before:.4f}")
print(f"Brier Score Setelah Kalibrasi : {brier_after:.4f} (Penurunan error probabilitas)")

# 9.3 Plot Calibration Curve (Reliability Diagram)
prob_true_raw, prob_pred_raw = calibration_curve(y_test, raw_probs, n_bins=5)
prob_true_cal, prob_pred_cal = calibration_curve(y_test, calib_probs, n_bins=5)

plt.figure(figsize=(7, 6))
plt.plot([0, 1], [0, 1], "k:", label="Kalibrasi Sempurna (Perfect)")
plt.plot(prob_pred_raw, prob_true_raw, "s-", color='#ef4444', label=f"{best_model_name} Mentah (Brier = {brier_before:.3f})")
plt.plot(prob_pred_cal, prob_true_cal, "o-", color='#10b981', label=f"{best_model_name} Terkalibrasi (Brier = {brier_after:.3f})")

plt.xlabel("Rata-rata Prediksi Probabilitas AI")
plt.ylabel("Proporsi Positif Aktual")
plt.title("Reliability Diagram: Kalibrasi Probabilitas Medis")
plt.legend(loc="lower right")
plt.show()



---
## STEP 10: Final Model Selection & Serialization

Kita membungkus `StandardScaler` dan `CalibratedClassifierCV` ke dalam satu objek **`Pipeline`** utuh. Dengan cara ini, backend GluCare cukup menerima data mentah pengguna (tanpa perlu manual scaling di sisi backend).  
Pipeline diekspor ke format **`.joblib`** bersama file metadata JSON.



In [ ]:
# 10.1 Membangun End-to-End Pipeline
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', CalibratedClassifierCV(estimator=best_base_model, method='sigmoid', cv=5))
])

# Fit pipeline pada seluruh Training Set
final_pipeline.fit(X_train, y_train)

# 10.2 Evaluasi Akhir Pipeline pada Held-out Test Set
final_preds = final_pipeline.predict(X_test)
final_probs = final_pipeline.predict_proba(X_test)[:, 1]

final_metrics = {
    'accuracy': float(accuracy_score(y_test, final_preds)),
    'recall': float(recall_score(y_test, final_preds)),
    'precision': float(precision_score(y_test, final_preds)),
    'f1': float(f1_score(y_test, final_preds)),
    'roc_auc': float(roc_auc_score(y_test, final_probs)),
    'brier_score': float(brier_score_loss(y_test, final_probs))
}

print("=== PERFORMA FINAL PIPELINE TERKALIBRASI PADA TEST SET ===")
for k, v in final_metrics.items():
    print(f"  * {k.upper():12s}: {v:.4f}")

# 10.3 Serialisasi Model ke Format .joblib
model_export_path = MODELS_DIR / 'glucare_clinical_calibrated_pipeline.joblib'
joblib.dump({
    'pipeline': final_pipeline,
    'model_name': f"{best_model_name} (Calibrated)",
    'features': FEATURE_COLS,
    'target_classes': {0: 'Bukan Prediabetes (Normal)', 1: 'Prediabetes'},
    'test_metrics': final_metrics
}, model_export_path)

# Simpan metadata JSON
metadata_path = MODELS_DIR / 'model_metadata_10_steps.json'
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump({
        'model_name': f"{best_model_name} (Calibrated)",
        'feature_names': FEATURE_COLS,
        'metrics': final_metrics,
        'train_samples': len(X_train),
        'test_samples': len(X_test)
    }, f, indent=2)

print(f"\nModel Pipeline berhasil diekspor ke: {model_export_path}")
print(f"Ukuran file model: {model_export_path.stat().st_size / 1024:.1f} KB")



---
## STEP 11: Feature Importance & Clinical Explainability (Faktor Pendorong Risiko)

Untuk mematuhi prinsip **Responsible AI & Healthcare Transparency**, kita mengekstraksi peringkat fitur klinis yang paling berkontribusi dalam menentukan risiko pasien.



In [ ]:
# 11.1 Ekstraksi Feature Importance dari Base Estimator
importances = None
if hasattr(best_base_model, 'feature_importances_'):
    importances = best_base_model.feature_importances_
elif hasattr(best_base_model, 'coef_'):
    importances = np.abs(best_base_model.coef_[0])

if importances is not None:
    feat_df = pd.DataFrame({
        'Fitur': FEATURE_COLS,
        'Kontribusi (%)': (importances / importances.sum()) * 100
    }).sort_values(by='Kontribusi (%)', ascending=False).reset_index(drop=True)
    
    plt.figure(figsize=(9, 5))
    sns.barplot(data=feat_df, x='Kontribusi (%)', y='Fitur', palette='viridis')
    plt.title('Peringkat Kepentingan Fitur Klinis (Feature Importance)')
    plt.xlabel('Kontribusi Terhadap Keputusan AI (%)')
    plt.tight_layout()
    plt.show()
    
    print("=== PERINGKAT KONTRIBUSI FITUR KLINIS ===")
    display(feat_df.round(2))



---
## STEP 12: Sanity Check & Diagnostic Simulation (Inference Testing)

Kita memuat kembali file artefak `.joblib` yang telah diekspor dan menguji respons AI terhadap **3 profil pasien simulasi**:
1. **Pasien A (Sehat / Normal)**: Parameter dalam batas optimal.
2. **Pasien B (Borderline)**: Glukosa normal tinggi, lingkar perut dan BMI berlebih.
3. **Pasien C (Risiko Tinggi Prediabetes)**: Glukosa puasa $\ge 105$ mg/dL, rasio TG/HDL tinggi.



In [ ]:
# 12.1 Load Model Artefak
loaded_artifact = joblib.load(model_export_path)
loaded_pipe = loaded_artifact['pipeline']

# 12.2 Tiga Skenario Pasien Simulasi
simulation_patients = [
    {
        'title': 'Pasien A (Dewasa Muda Sehat & Aktif)',
        'data': {
            'age': 25, 'gender': 1, 'weight': 62.0, 'bmi': 21.5, 'waist_cm': 75.0,
            'bp_systolic': 110.0, 'bp_diastolic': 70.0, 'map_pressure': 83.33,
            'glucose_fasting': 88.0, 'hdl': 58.0, 'triglycerides': 75.0, 'tg_hdl_ratio': 1.29
        }
    },
    {
        'title': 'Pasien B (Borderline / Gejala Awal Overweight)',
        'data': {
            'age': 32, 'gender': 2, 'weight': 78.0, 'bmi': 27.2, 'waist_cm': 89.0,
            'bp_systolic': 122.0, 'bp_diastolic': 80.0, 'map_pressure': 94.00,
            'glucose_fasting': 98.0, 'hdl': 46.0, 'triglycerides': 130.0, 'tg_hdl_ratio': 2.82
        }
    },
    {
        'title': 'Pasien C (Risiko Tinggi Prediabetes & Dislipidemia)',
        'data': {
            'age': 38, 'gender': 1, 'weight': 92.0, 'bmi': 31.5, 'waist_cm': 104.0,
            'bp_systolic': 135.0, 'bp_diastolic': 88.0, 'map_pressure': 103.67,
            'glucose_fasting': 112.0, 'hdl': 38.0, 'triglycerides': 190.0, 'tg_hdl_ratio': 5.00
        }
    }
]

print("=== HASIL SANITY CHECK INFERENSI PADA 3 SKENARIO PASIEN ===\n")
for case in simulation_patients:
    df_single = pd.DataFrame([case['data']])[FEATURE_COLS]
    pred = loaded_pipe.predict(df_single)[0]
    probs = loaded_pipe.predict_proba(df_single)[0]
    risk_score = probs[1] * 100
    
    status = "PREDIABETES (High Risk)" if pred == 1 else "NORMAL (Low Risk)"
    
    print(f">> {case['title']}")
    print(f"   Status AI      : {status}")
    print(f"   Skor Risiko    : {risk_score:.1f}% (Probabilitas Prediabetes)")
    print(f"   Prob. Normal   : {probs[0]*100:.1f}%")
    print(f"   Rekomendasi    : {'Disarankan intervensi gaya hidup Program 90 Hari GluCare.' if pred == 1 else 'Pertahankan pola hidup sehat dan kontrol berkala.'}")
    print("-" * 75)

